In [2]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (roc_auc_score, precision_score, recall_score, 
                           f1_score, accuracy_score, confusion_matrix)
import warnings
warnings.filterwarnings('ignore')

class ThreeModelsValidation:
    """
    Final validation of three models with hold-out test sets
    """
    
    def __init__(self):
        # Model configurations based on best results do cross-validation
        self.model_configs = {
            'Model_1': {
                'name': 'Model 1 (Demographic/Perinatal)',
                'algorithm': 'Random Forest',
                'train_path': '/Users/marcelosilva/Desktop/early-obesity-prediction/D-Train-Test Split/MODEL1TRAIN.csv',
                'test_path': '/Users/marcelosilva/Desktop/early-obesity-prediction/D-Train-Test Split/MODEL1TEST.csv',
                'expected_auc': 0.570,
                'expected_ci': (0.529, 0.611)
            },
            'Model_2': {
                'name': 'Model 2 (Feeding Practices)',
                'algorithm': 'Logistic Regression',
                'train_path': '/Users/marcelosilva/Desktop/early-obesity-prediction/D-Train-Test Split/MODEL2TRAIN.csv',
                'test_path': '/Users/marcelosilva/Desktop/early-obesity-prediction/D-Train-Test Split/MODEL2TEST.csv',
                'expected_auc': 0.598,
                'expected_ci': (0.504, 0.692)
            },
            'Model_3': {
                'name': 'Model 3 (Combined)',
                'algorithm': 'Gradient Boosting',
                'train_path': '/Users/marcelosilva/Desktop/early-obesity-prediction/D-Train-Test Split/MODEL3TRAIN.csv',
                'test_path': '/Users/marcelosilva/Desktop/early-obesity-prediction/D-Train-Test Split/MODEL3TEST.csv',
                'expected_auc': 0.571,
                'expected_ci': (0.503, 0.639)
            }
        }
        
        # Binary variables that need fixing de tipo
        self.binary_vars_model2_3 = [
            'doou_leite_banco', 'recebeu_leite_banco', 'amamentou_outra_crianca',
            'usou_mamadeira', 'deixou_amamentar_por_outra', 'busca_info_aleitamento',
            'deu_outros_liquidos', 'k15_recebeu', 'k11_amamentou', 'k03_prenatal',
            'usou_utensilios_amamentacao'
        ]
        
        self.results = {}
    
    def create_model_pipeline(self, algorithm, random_state=42):
        """
        Creates model pipeline based on specified algorithm
        """
        if algorithm == 'Random Forest':
            # Optimized hyperparameters for Random Forest
            model = Pipeline([
                ('scaler', RobustScaler()),
                ('model', RandomForestClassifier(
                    n_estimators=100,
                    max_depth=10,
                    min_samples_split=5,
                    min_samples_leaf=2,
                    random_state=random_state,
                    class_weight='balanced',
                    n_jobs=-1
                ))
            ])
        
        elif algorithm == 'Logistic Regression':
            # Optimized hyperparameters for Logistic Regression (Ridge)
            model = Pipeline([
                ('scaler', RobustScaler()),
                ('model', LogisticRegression(
                    random_state=random_state,
                    max_iter=10000,
                    penalty='l2',  # Ridge
                    C=100.0,
                    class_weight='balanced'
                ))
            ])
        
        elif algorithm == 'Gradient Boosting':
            # Optimized hyperparameters for Gradient Boosting
            model = Pipeline([
                ('scaler', RobustScaler()),
                ('model', GradientBoostingClassifier(
                    n_estimators=100,
                    learning_rate=0.1,
                    max_depth=3,
                    random_state=random_state
                ))
            ])
        
        else:
            raise ValueError(f"Algorithm not recognized: {algorithm}")
        
        return model
    
    def calculate_metrics(self, y_true, y_pred, y_pred_proba):
        """
        Calculates all performance metrics
        """
        metrics = {}
        
        # Basic metrics
        metrics['auc'] = roc_auc_score(y_true, y_pred_proba)
        metrics['precision'] = precision_score(y_true, y_pred, zero_division=0)
        metrics['recall'] = recall_score(y_true, y_pred, zero_division=0)
        metrics['f1'] = f1_score(y_true, y_pred, zero_division=0)
        metrics['accuracy'] = accuracy_score(y_true, y_pred)
        
        # Confusion matrix for specificity and NPV
        cm = confusion_matrix(y_true, y_pred)
        if cm.shape == (2, 2):
            tn, fp, fn, tp = cm.ravel()
            metrics['specificity'] = tn / (tn + fp) if (tn + fp) > 0 else 0
            metrics['npv'] = tn / (tn + fn) if (tn + fn) > 0 else 0
            metrics['ppv'] = tp / (tp + fp) if (tp + fp) > 0 else 0  # Mesmo que precision
        else:
            # Special case: only one class present
            if y_true.sum() == 0:  # Only negatives
                metrics['specificity'] = 1.0
                metrics['npv'] = 1.0
                metrics['ppv'] = 0.0
            else:  # Only positives
                metrics['specificity'] = 0.0
                metrics['npv'] = 0.0
                metrics['ppv'] = 1.0
        
        return metrics
    
    def bootstrap_confidence_intervals(self, y_true, y_pred, y_pred_proba, n_bootstrap=1000):
        """
        Calculates 95% confidence intervals using bootstrap
        """
        np.random.seed(42)
        
        bootstrap_metrics = {
            'auc': [], 'precision': [], 'recall': [], 'f1': [],
            'accuracy': [], 'specificity': [], 'npv': [], 'ppv': []
        }
        
        for i in range(n_bootstrap):
            # Bootstrap sampling
            indices = np.random.choice(len(y_true), size=len(y_true), replace=True)
            y_true_boot = y_true.iloc[indices] if hasattr(y_true, 'iloc') else y_true[indices]
            y_pred_boot = y_pred[indices]
            y_pred_proba_boot = y_pred_proba[indices]
            
            try:
                metrics_boot = self.calculate_metrics(y_true_boot, y_pred_boot, y_pred_proba_boot)
                
                for metric, value in metrics_boot.items():
                    bootstrap_metrics[metric].append(value)
                    
            except Exception:
                # In case of error, use original metrics
                original_metrics = self.calculate_metrics(y_true, y_pred, y_pred_proba)
                for metric, value in original_metrics.items():
                    bootstrap_metrics[metric].append(value)
        
        # Calcular intervalos de confiança
        ci_results = {}
        for metric, values in bootstrap_metrics.items():
            mean_val = np.mean(values)
            ci_lower = np.percentile(values, 2.5)
            ci_upper = np.percentile(values, 97.5)
            
            ci_results[metric] = {
                'mean': mean_val,
                'ci_lower': ci_lower,
                'ci_upper': ci_upper
            }
        
        return ci_results
    
    def validate_single_model(self, model_key):
        """
        Validates a specific model
        """
        config = self.model_configs[model_key]
        
        print(f"\n{'='*80}")
        print(f"FINAL VALIDATION - {config['name'].upper()}")
        print(f"{'='*80}")
        
        # Load data
        df_train = pd.read_csv(config['train_path'])
        df_test = pd.read_csv(config['test_path'])
        
        # Fix data types para modelos 2 e 3
        if model_key in ['Model_2', 'Model_3']:
            for var in self.binary_vars_model2_3:
                if var in df_train.columns:
                    df_train[var] = df_train[var].astype(int)
                if var in df_test.columns:
                    df_test[var] = df_test[var].astype(int)
        
        # Prepare data
        target_cols = ['desnutrido', 'eutrofico', 'sobrepeso', 'obeso']
        
        y_train = df_train['obeso'].copy()
        X_train = df_train.drop(target_cols + ['id_anon'], axis=1)
        
        y_test = df_test['obeso'].copy()
        X_test = df_test.drop(target_cols + ['id_anon'], axis=1)
        
        print(f"Training data:")
        print(f"  - Observations: {len(df_train):,}")
        print(f"  - Features: {X_train.shape[1]}")
        print(f"  - Obese: {y_train.sum()} ({y_train.mean()*100:.1f}%)")
        
        print(f"\nTest data:")
        print(f"  - Observations: {len(df_test):,}")
        print(f"  - Features: {X_test.shape[1]}")
        print(f"  - Obese: {y_test.sum()} ({y_test.mean()*100:.1f}%)")
        
        # Criar e treinar modelo
        model = self.create_model_pipeline(config['algorithm'])
        
        print(f"\nTraining final model...")
        print(f"  - Algorithm: {config['algorithm']}")
        print(f"  - Preprocessing: RobustScaler")
        print(f"  - Balancing: class_weight='balanced'")
        
        # Train model
        model.fit(X_train, y_train)
        
        # Test set predictions
        y_pred_proba = model.predict_proba(X_test)[:, 1]
        y_pred = model.predict(X_test)
        
        # Calcular métricas basic
        base_metrics = self.calculate_metrics(y_test, y_pred, y_pred_proba)
        
        print(f"\nTest set results (single run):")
        for metric, value in base_metrics.items():
            print(f"  - {metric.upper()}: {value:.3f}")
        
        # Bootstrap para intervalos de confiança
        print(f"\nCalculating bootstrap confidence intervals (n=1,000)...")
        
        ci_results = self.bootstrap_confidence_intervals(y_test, y_pred, y_pred_proba)
        
        # Armazenar resultados
        self.results[model_key] = {
            'config': config,
            'base_metrics': base_metrics,
            'ci_results': ci_results,
            'model': model
        }
        
        return ci_results
    
    def print_results_table(self, model_key):
        """
        Prints formatted results table
        """
        config = self.model_configs[model_key]
        ci_results = self.results[model_key]['ci_results']
        
        print(f"\n{'='*90}")
        print(f"FINAL RESULTS WITH 95% CI - {config['name'].upper()}")
        print(f"{'='*90}")
        
        metrics_names = {
            'auc': 'AUC-ROC',
            'precision': 'Precision',
            'recall': 'Recall (Sensitivity)',
            'f1': 'F1-Score',
            'accuracy': 'Accuracy',
            'specificity': 'Specificity',
            'npv': 'NPV',
            'ppv': 'PPV'
        }
        
        print(f"{'Metric':<25} {'Value':<10} {'IC 95%':<25} {'Interpretation':<25}")
        print("-" * 90)
        
        for metric, name in metrics_names.items():
            if metric in ci_results:
                result = ci_results[metric]
                mean_val = result['mean']
                ci_lower = result['ci_lower']
                ci_upper = result['ci_upper']
                
                # Interpretation especial para AUC-ROC
                if metric == 'auc':
                    if ci_lower <= 0.5:
                        interpretation = "Includes chance (≤0.5)"
                    elif ci_lower > 0.5 and ci_upper < 0.7:
                        interpretation = "Weak (>0.5, <0.7)"
                    else:
                        interpretation = "Moderate (≥0.7)"
                else:
                    interpretation = ""
                
                ci_str = f"({ci_lower:.3f}-{ci_upper:.3f})"
                print(f"{name:<25} {mean_val:<10.3f} {ci_str:<25} {interpretation:<25}")
    
    def critical_analysis(self, model_key):
        """
        Critical analysis of results
        """
        config = self.model_configs[model_key]
        auc_result = self.results[model_key]['ci_results']['auc']
        precision_result = self.results[model_key]['ci_results']['precision']
        
        print(f"\n{'='*70}")
        print(f"CRITICAL ANALYSIS - {config['name'].upper()}")
        print(f"{'='*70}")
        
        print(f"Predictive Capacity:")
        if auc_result['ci_lower'] <= 0.5:
            print(f"  [FAIL] AUC IC 95% inclui ≤0.5: Does NOT statistically exceed chance")
            print(f"     Lower bound: {auc_result['ci_lower']:.3f}")
            print(f"     Difference from chance: {auc_result['ci_lower'] - 0.5:.3f}")
        else:
            print(f"  [OK] AUC IC 95% exclui ≤0.5: Statistically exceeds chance")
            print(f"     Lower bound: {auc_result['ci_lower']:.3f}")
        
        print(f"\nClinical Utility:")
        print(f"  - Precision {precision_result['mean']:.1%}: {100-precision_result['mean']*100:.1f}% false positives")
        
        if precision_result['mean'] < 0.1:
            print(f"  [FAIL] Precision < 10%: Clinically useless")
        
        print(f"\nComparison with Cross-Validation:")
        print(f"  - Expected CV AUC: {config['expected_auc']:.3f}")
        print(f"  - Hold-out AUC: {auc_result['mean']:.3f}")
        print(f"  - Difference: {auc_result['mean'] - config['expected_auc']:.3f}")
    
    def run_all_validations(self):
        """
        Executes validation of all three models
        """
        print("FINAL VALIDATION OF THREE CHILDHOOD OBESITY PREDICTION MODELS")
        print("="*80)
        print("Objective: Confirm that no model statistically exceeds chance")
        
        # Validar cada modelo
        for model_key in ['Model_1', 'Model_2', 'Model_3']:
            self.validate_single_model(model_key)
            self.print_results_table(model_key)
            self.critical_analysis(model_key)
        
        # Resumo comparativo final
        self.comparative_summary()
    
    def comparative_summary(self):
        """
        Final comparative summary of three models
        """
        print(f"\n{'='*80}")
        print("FINAL COMPARATIVE SUMMARY")
        print(f"{'='*80}")
        
        print(f"{'Model':<30} {'Algorithm':<20} {'AUC-ROC':<15} {'IC 95%':<20} {'Exceeds Chance?'}")
        print("-" * 90)
        
        for model_key in ['Model_1', 'Model_2', 'Model_3']:
            config = self.results[model_key]['config']
            auc_result = self.results[model_key]['ci_results']['auc']
            
            auc_mean = auc_result['mean']
            ci_str = f"({auc_result['ci_lower']:.3f}-{auc_result['ci_upper']:.3f})"
            supera_acaso = "NO" if auc_result['ci_lower'] <= 0.5 else "YES"
            
            print(f"{config['name']:<30} {config['algorithm']:<20} {auc_mean:<15.3f} {ci_str:<20} {supera_acaso}")
        
        print(f"\n{'='*80}")
        print("GENERAL SCIENTIFIC CONCLUSION")
        print(f"={'='*80}")
        print("ROBUST EVIDENCE OF PREDICTIVE LIMITATIONS:")
        print("  - ALL models failed to statistically exceed chance")
        print("  - First 24 months factors inadequate for obesity prediction")
        print("  - Hold-out validation confirms negative cross-validation results")
        print("  - Precision consistently <5% indicates high false positive rate")
        
        print(f"\nImplications for Research and Policy:")
        print("  - Early prediction paradigm should be questioned")
        print("  - Focus should shift to modifiable proximal factors")
        print("  - Interventions based on these predictors would be ineffective")
        print("  - Negative results are scientifically valuable")

# Execute validation completa
if __name__ == "__main__":
    validator = ThreeModelsValidation()
    validator.run_all_validations()

FINAL VALIDATION OF THREE CHILDHOOD OBESITY PREDICTION MODELS
Objective: Confirm that no model statistically exceeds chance

FINAL VALIDATION - MODEL 1 (DEMOGRAPHIC/PERINATAL)
Training data:
  - Observations: 6,588
  - Features: 27
  - Obese: 183 (2.8%)

Test data:
  - Observations: 1,648
  - Features: 27
  - Obese: 46 (2.8%)

Training final model...
  - Algorithm: Random Forest
  - Preprocessing: RobustScaler
  - Balancing: class_weight='balanced'

Test set results (single run):
  - AUC: 0.598
  - PRECISION: 0.000
  - RECALL: 0.000
  - F1: 0.000
  - ACCURACY: 0.971
  - SPECIFICITY: 0.999
  - NPV: 0.972
  - PPV: 0.000

Calculating bootstrap confidence intervals (n=1,000)...

FINAL RESULTS WITH 95% CI - MODEL 1 (DEMOGRAPHIC/PERINATAL)
Metric                   Value      IC 95%                    Interpretation            
------------------------------------------------------------------------------------------
AUC-ROC                   0.597      (0.517-0.672)             Weak (>0.5, <